<a href="https://colab.research.google.com/github/KaelynWen/Clothy/blob/main/batch/AlphaFold2_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ColabFold v1.6.0: AlphaFold2 w/ MMseqs2 BATCH

<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="256" align="right" style="height:256px">

Easy to use AlphaFold2 protein structure [(Jumper et al. 2021)](https://www.nature.com/articles/s41586-021-03819-2) and complex [(Evans et al. 2021)](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1) prediction using multiple sequence alignments generated through MMseqs2. For details, refer to our manuscript:

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all.
*Nature Methods*, 2022](https://www.nature.com/articles/s41592-022-01488-1)

**Usage**

`input_dir` directory with only fasta files or MSAs stored in Google Drive. MSAs need to be A3M formatted and have an `.a3m` extention. For MSAs MMseqs2 will not be called.

`result_dir` results will be written to the result directory in Google Drive

Old versions: [v1.4](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.4.0/batch/AlphaFold2_batch.ipynb), [v1.5.1](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.1/batch/AlphaFold2_batch.ipynb), [v1.5.2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.2/batch/AlphaFold2_batch.ipynb), [v1.5.3-patch](https://colab.research.google.com/github/sokrypton/ColabFold/blob/56c72044c7d51a311ca99b953a71e552fdc042e1/batch/AlphaFold2_batch.ipynb)

<strong>For more details, see <a href="#Instructions">bottom</a> of the notebook and checkout the [ColabFold GitHub](https://github.com/sokrypton/ColabFold). </strong>

-----------

### News
- <b><font color='green'>2023/07/31: The ColabFold MSA server is back to normal. It was using older DB (UniRef30 2202/PDB70 220313) from 27th ~8:30 AM CEST to 31st ~11:10 AM CEST.</font></b>
- <b><font color='green'>2023/06/12: New databases! UniRef30 updated to 2023_02 and PDB to 230517. We now use PDB100 instead of PDB70 (see notes in the [main](https://colabfold.com) notebook).</font></b>
- <b><font color='green'>2023/06/12: We introduced a new default pairing strategy: Previously, for multimer predictions with more than 2 chains, we only pair if all sequences taxonomically match ("complete" pairing). The new default "greedy" strategy pairs any taxonomically matching subsets.</font></b>

In [21]:
#@title Mount google drive
from google.colab import drive
drive.mount('/content/drive')
from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import os
os.makedirs('/content/drive/MyDrive/input_fasta', exist_ok=True)
os.makedirs('/content/drive/MyDrive/result', exist_ok=True)

In [24]:
import os

input_dir = "/content/drive/MyDrive/input_fasta"
os.makedirs(input_dir, exist_ok=True)

seqs = {
    "hLcn2.fasta": """>hLcn2
QDSTSDLIPAPPLSKVPLQQNFQDNQFQGKWYVVGLAGNAILREDKDPQKMYATIYELKEDKSYNVTSVLFRKKKCDYWIRTFVPGCQPGEFTLGNIKSYPGLTSYLVRVVSTNYNQHAMVFFKKVSQNREYFKITLYGRTKELTSELKENFIRFSKSLGLPENHIVFPVPIDQCIDG
""",
    "C87A.fasta": """>C87A
QDSTSDLIPAPPLSKVPLQQNFQDNQFQGKWYVVGLAGNAILREDKDPQKMYATIYELKEDKSYNVTSVLFRKKKCDYWIRTFVPGAQPGEFTLGNIKSYPGLTSYLVRVVSTNYNQHAMVFFKKVSQNREYFKITLYGRTKELTSELKENFIRFSKSLGLPENHIVFPVPIDQCIDG
""",
    "R81E.fasta": """>R81E
QDSTSDLIPAPPLSKVPLQQNFQDNQFQGKWYVVGLAGNAILREDKDPQKMYATIYELKEDKSYNVTSVLFRKKKCDYWIETFVPGCQPGEFTLGNIKSYPGLTSYLVRVVSTNYNQHAMVFFKKVSQNREYFKITLYGRTKELTSELKENFIRFSKSLGLPENHIVFPVPIDQCIDG
"""
}

for fname, content in seqs.items():
    with open(os.path.join(input_dir, fname), "w") as f:
        f.write(content)

print("Saved 3 separate FASTA files.")
print(os.listdir(input_dir))

Saved 3 separate FASTA files.
['lcn2_variants.fasta', 'hLcn2.fasta', 'C87A.fasta', 'R81E.fasta']


In [25]:
!ls -lh /content/drive/MyDrive/input_fasta

total 2.5K
-rw------- 1 root root 185 Mar 15 08:25 C87A.fasta
-rw------- 1 root root 186 Mar 15 08:25 hLcn2.fasta
-rw------- 1 root root 558 Mar 15 08:15 lcn2_variants.fasta
-rw------- 1 root root 185 Mar 15 08:25 R81E.fasta


In [26]:
!for f in /content/drive/MyDrive/input_fasta/*.fasta; do echo "==== $f ===="; head -5 "$f"; echo; done

==== <_io.TextIOWrapper name='/content/drive/MyDrive/input_fasta/R81E.fasta' mode='w' encoding='utf-8'> ====
head: cannot open '<_io.TextIOWrapper name='\''/content/drive/MyDrive/input_fasta/R81E.fasta'\'' mode='\''w'\'' encoding='\''utf-8'\''>' for reading: No such file or directory

==== <_io.TextIOWrapper name='/content/drive/MyDrive/input_fasta/R81E.fasta' mode='w' encoding='utf-8'> ====
head: cannot open '<_io.TextIOWrapper name='\''/content/drive/MyDrive/input_fasta/R81E.fasta'\'' mode='\''w'\'' encoding='\''utf-8'\''>' for reading: No such file or directory

==== <_io.TextIOWrapper name='/content/drive/MyDrive/input_fasta/R81E.fasta' mode='w' encoding='utf-8'> ====
head: cannot open '<_io.TextIOWrapper name='\''/content/drive/MyDrive/input_fasta/R81E.fasta'\'' mode='\''w'\'' encoding='\''utf-8'\''>' for reading: No such file or directory

==== <_io.TextIOWrapper name='/content/drive/MyDrive/input_fasta/R81E.fasta' mode='w' encoding='utf-8'> ====
head: cannot open '<_io.TextIOWra

In [27]:
input_dir = '/content/drive/MyDrive/input_fasta'
result_dir = '/content/drive/MyDrive/result'
msa_mode = "MMseqs2 (UniRef+Environmental)"
num_models = 5
num_recycles = 3
stop_at_score = 100
num_relax = 0
relax_max_iterations = 200
use_templates = False
do_not_overwrite_results = True
zip_results = False

In [28]:
zip_results = True

In [29]:
#@title Input protein sequence, then hit `Runtime` -> `Run all`

input_dir = '/content/drive/MyDrive/input_fasta' #@param {type:"string"}
result_dir = '/content/drive/MyDrive/result' #@param {type:"string"}

# number of models to use
#@markdown ---
#@markdown ### Advanced settings
msa_mode = "MMseqs2 (UniRef+Environmental)" #@param ["MMseqs2 (UniRef+Environmental)", "MMseqs2 (UniRef only)","single_sequence","custom"]
num_models = 5 #@param [1,2,3,4,5] {type:"raw"}
num_recycles = 3 #@param [1,3,6,12,24,48] {type:"raw"}
stop_at_score = 100 #@param {type:"string"}
#@markdown - early stop computing models once score > threshold (avg. plddt for "structures" and ptmscore for "complexes")
use_custom_msa = False
num_relax = 0 #@param [0, 1, 5] {type:"raw"}
use_amber = num_relax > 0
relax_max_iterations = 200 #@param [0,200,2000] {type:"raw"}
use_templates = False #@param {type:"boolean"}
do_not_overwrite_results = True #@param {type:"boolean"}
zip_results = False #@param {type:"boolean"}


In [30]:
#@title Install dependencies
%%bash -s $use_amber $use_templates $python_version

set -e

USE_AMBER=$1
USE_TEMPLATES=$2
PYTHON_VERSION=$3

if [ ! -f COLABFOLD_READY ]; then
  # install dependencies
  # We have to use "--no-warn-conflicts" because colab already has a lot preinstalled with requirements different to ours
  pip install -q --no-warn-conflicts "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
  if [ -n "${TPU_NAME}" ]; then
    pip install -q --no-warn-conflicts -U dm-haiku==0.0.10 jax==0.3.25
  fi
  ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold
  ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold
  # hack to fix TF crash
  rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so
  touch COLABFOLD_READY
fi

# Download params (~1min)
python -m colabfold.download

# setup conda
if [ ${USE_AMBER} == "True" ] || [ ${USE_TEMPLATES} == "True" ]; then
  if [ ! -f CONDA_READY ]; then
    wget -qnc https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-25.3.1-0-Linux-x86_64.sh
    bash Miniforge3-25.3.1-0-Linux-x86_64.sh -bfp /usr/local 2>&1 1>/dev/null
    rm Miniforge3-25.3.1-0-Linux-x86_64.sh
    conda config --set auto_update_conda false
    touch CONDA_READY
  fi
fi
# setup template search
if [ ${USE_TEMPLATES} == "True" ] && [ ! -f HH_READY ]; then
  conda install -y -q -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python="${PYTHON_VERSION}" 2>&1 1>/dev/null
  touch HH_READY
fi
# setup openmm for amber refinement
if [ ${USE_AMBER} == "True" ] && [ ! -f AMBER_READY ]; then
  conda install -y -q -c conda-forge openmm=8.2.0 python="${PYTHON_VERSION}" pdbfixer 2>&1 1>/dev/null
  touch AMBER_READY
fi

In [31]:
#@title Run Prediction

import sys

from colabfold.batch import get_queries, run
from colabfold.download import default_data_dir
from colabfold.utils import setup_logging
from pathlib import Path

# For some reason we need that to get pdbfixer to import
if use_amber and f"/usr/local/lib/python{python_version}/site-packages/" not in sys.path:
    sys.path.insert(0, f"/usr/local/lib/python{python_version}/site-packages/")

setup_logging(Path(result_dir).joinpath("log.txt"))

queries, is_complex = get_queries(input_dir)
run(
    queries=queries,
    result_dir=result_dir,
    use_templates=use_templates,
    num_relax=num_relax,
    relax_max_iterations=relax_max_iterations,
    msa_mode=msa_mode,
    model_type="auto",
    num_models=num_models,
    num_recycles=num_recycles,
    model_order=[1, 2, 3, 4, 5],
    is_complex=is_complex,
    data_dir=default_data_dir,
    keep_existing_results=do_not_overwrite_results,
    rank_by="auto",
    pair_mode="unpaired+paired",
    stop_at_score=stop_at_score,
    zip_results=zip_results,
    user_agent="colabfold/google-colab-batch",
)

2026-03-15 08:25:38,165 More than one sequence in /content/drive/MyDrive/input_fasta/lcn2_variants.fasta, ignoring all but the first sequence
2026-03-15 08:25:38,167 Running on GPU
2026-03-15 08:25:38,183 Found 5 citations for tools or databases
2026-03-15 08:25:38,184 Query 1/4: C87A (length 178)


PENDING:   0%|          | 0/150 [elapsed: 00:00 remaining: ?]

2026-03-15 08:25:38,953 Sleeping for 8s. Reason: PENDING


RUNNING:   5%|▌         | 8/150 [elapsed: 00:09 remaining: 02:47]

2026-03-15 08:25:47,650 Sleeping for 9s. Reason: RUNNING


...

COMPLETE: 100%|██████████| 150/150 [elapsed: 00:45 remaining: 00:00]


2026-03-15 08:32:43,310 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=92.7 pTM=0.88
2026-03-15 08:32:56,500 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=94.1 pTM=0.89 tol=0.374
2026-03-15 08:33:09,800 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=94.2 pTM=0.891 tol=0.0836
2026-03-15 08:33:22,928 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=94.2 pTM=0.89 tol=0.0529
2026-03-15 08:33:22,929 alphafold2_ptm_model_1_seed_000 took 52.6s (3 recycles)
2026-03-15 08:33:35,928 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=92.4 pTM=0.884
2026-03-15 08:33:48,866 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=94.2 pTM=0.898 tol=0.351
2026-03-15 08:34:01,799 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=94.3 pTM=0.898 tol=0.125
2026-03-15 08:34:14,778 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=94.4 pTM=0.898 tol=0.0726
2026-03-15 08:34:14,779 alphafold2_ptm_model_2_seed_000 took 51.8s (3 recycles)
2026-03-15 08:34:27,914 alphafold2_ptm_model_3_seed_000 recycle=0 pLDDT=93.5 pTM=0.8

COMPLETE: 100%|██████████| 150/150 [elapsed: 00:01 remaining: 00:00]


2026-03-15 08:37:11,178 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=92.6 pTM=0.876
2026-03-15 08:37:24,349 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=94.1 pTM=0.889 tol=0.287
2026-03-15 08:37:37,539 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=94.1 pTM=0.889 tol=0.086
2026-03-15 08:37:50,652 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=94.2 pTM=0.89 tol=0.106
2026-03-15 08:37:50,654 alphafold2_ptm_model_1_seed_000 took 52.5s (3 recycles)
2026-03-15 08:38:03,748 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=92.4 pTM=0.881
2026-03-15 08:38:16,751 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=94.1 pTM=0.896 tol=0.278
2026-03-15 08:38:29,783 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=94.1 pTM=0.896 tol=0.0814
2026-03-15 08:38:42,775 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=94.1 pTM=0.896 tol=0.0547
2026-03-15 08:38:42,776 alphafold2_ptm_model_2_seed_000 took 52.1s (3 recycles)
2026-03-15 08:38:55,829 alphafold2_ptm_model_3_seed_000 recycle=0 pLDDT=93.3 pTM=0.

{'rank': [['rank_001_alphafold2_ptm_model_5_seed_000',
   'rank_002_alphafold2_ptm_model_3_seed_000',
   'rank_003_alphafold2_ptm_model_4_seed_000',
   'rank_004_alphafold2_ptm_model_1_seed_000',
   'rank_005_alphafold2_ptm_model_2_seed_000'],
  ['rank_001_alphafold2_ptm_model_3_seed_000',
   'rank_002_alphafold2_ptm_model_4_seed_000',
   'rank_003_alphafold2_ptm_model_5_seed_000',
   'rank_004_alphafold2_ptm_model_2_seed_000',
   'rank_005_alphafold2_ptm_model_1_seed_000'],
  ['rank_001_alphafold2_ptm_model_5_seed_000',
   'rank_002_alphafold2_ptm_model_3_seed_000',
   'rank_003_alphafold2_ptm_model_4_seed_000',
   'rank_004_alphafold2_ptm_model_1_seed_000',
   'rank_005_alphafold2_ptm_model_2_seed_000']],
 'metric': [[{'mean_plddt': 95.4375,
    'ptm': 0.90380859375,
    'print_line': ' pLDDT=95.4 pTM=0.904'},
   {'mean_plddt': 95.3125,
    'ptm': 0.904296875,
    'print_line': ' pLDDT=95.3 pTM=0.904'},
   {'mean_plddt': 95.125,
    'ptm': 0.9033203125,
    'print_line': ' pLDDT=95.1

In [32]:
result_dir = '/content/drive/MyDrive/result'

In [33]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive/result'):
    for f in files:
        if f.endswith('.pdb'):
            print(os.path.join(root, f))

/content/drive/MyDrive/result/lcn2_variants_unrelaxed_rank_004_alphafold2_ptm_model_1_seed_000.pdb
/content/drive/MyDrive/result/lcn2_variants_unrelaxed_rank_005_alphafold2_ptm_model_2_seed_000.pdb
/content/drive/MyDrive/result/lcn2_variants_unrelaxed_rank_002_alphafold2_ptm_model_3_seed_000.pdb
/content/drive/MyDrive/result/lcn2_variants_unrelaxed_rank_003_alphafold2_ptm_model_4_seed_000.pdb
/content/drive/MyDrive/result/lcn2_variants_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb
/content/drive/MyDrive/result/C87A_unrelaxed_rank_004_alphafold2_ptm_model_1_seed_000.pdb
/content/drive/MyDrive/result/C87A_unrelaxed_rank_005_alphafold2_ptm_model_2_seed_000.pdb
/content/drive/MyDrive/result/C87A_unrelaxed_rank_002_alphafold2_ptm_model_3_seed_000.pdb
/content/drive/MyDrive/result/C87A_unrelaxed_rank_003_alphafold2_ptm_model_4_seed_000.pdb
/content/drive/MyDrive/result/C87A_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb
/content/drive/MyDrive/result/R81E_unrelaxed_rank_005_a

In [34]:
load hLcn2_rank_001.pdb, wt
load C87A_rank_001.pdb, c87a
load R81E_rank_001.pdb, r81e

align c87a, wt
align r81e, wt

SyntaxError: invalid syntax (2789231525.py, line 1)

In [35]:
import py3Dmol

with open("/content/drive/MyDrive/result/hLcn2_rank_001.pdb", "r") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=600, height=400)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {}})
view.zoomTo()
view.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/result/hLcn2_rank_001.pdb'

In [36]:
import py3Dmol

files = {
    "wt": "/content/drive/MyDrive/result/hLcn2_rank_001.pdb",
    "c87a": "/content/drive/MyDrive/result/C87A_rank_001.pdb",
    "r81e": "/content/drive/MyDrive/result/R81E_rank_001.pdb"
}

view = py3Dmol.view(width=800, height=600)

for name, path in files.items():
    with open(path, "r") as f:
        pdb_data = f.read()
    view.addModel(pdb_data, "pdb")

view.setStyle({"model": 0}, {"cartoon": {}})
view.setStyle({"model": 1}, {"cartoon": {}})
view.setStyle({"model": 2}, {"cartoon": {}})
view.zoomTo()
view.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/result/hLcn2_rank_001.pdb'

In [37]:
import os

root = "/content/drive/MyDrive/result"

for dirpath, dirnames, filenames in os.walk(root):
    for f in filenames:
        if f.endswith(".pdb"):
            print(os.path.join(dirpath, f))

/content/drive/MyDrive/result/lcn2_variants_unrelaxed_rank_004_alphafold2_ptm_model_1_seed_000.pdb
/content/drive/MyDrive/result/lcn2_variants_unrelaxed_rank_005_alphafold2_ptm_model_2_seed_000.pdb
/content/drive/MyDrive/result/lcn2_variants_unrelaxed_rank_002_alphafold2_ptm_model_3_seed_000.pdb
/content/drive/MyDrive/result/lcn2_variants_unrelaxed_rank_003_alphafold2_ptm_model_4_seed_000.pdb
/content/drive/MyDrive/result/lcn2_variants_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb
/content/drive/MyDrive/result/C87A_unrelaxed_rank_004_alphafold2_ptm_model_1_seed_000.pdb
/content/drive/MyDrive/result/C87A_unrelaxed_rank_005_alphafold2_ptm_model_2_seed_000.pdb
/content/drive/MyDrive/result/C87A_unrelaxed_rank_002_alphafold2_ptm_model_3_seed_000.pdb
/content/drive/MyDrive/result/C87A_unrelaxed_rank_003_alphafold2_ptm_model_4_seed_000.pdb
/content/drive/MyDrive/result/C87A_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb
/content/drive/MyDrive/result/R81E_unrelaxed_rank_005_a

In [38]:
 import py3Dmol

files = {
    "wt": "/content/drive/MyDrive/result/hLcn2_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb",
    "c87a": "/content/drive/MyDrive/result/C87A_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb",
    "r81e": "/content/drive/MyDrive/result/R81E_unrelaxed_rank_001_alphafold2_ptm_model_3_seed_000.pdb"
}

view = py3Dmol.view(width=900, height=600)

for name, path in files.items():
    with open(path, "r") as f:
        pdb_data = f.read()
    view.addModel(pdb_data, "pdb")

view.setStyle({"model": 0}, {"cartoon": {"color": "blue"}})
view.setStyle({"model": 1}, {"cartoon": {"color": "red"}})
view.setStyle({"model": 2}, {"cartoon": {"color": "green"}})

view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [39]:
import py3Dmol

pdb_path = "/content/drive/MyDrive/result/hLcn2_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb"

with open(pdb_path, "r") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=700, height=500)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [40]:
import py3Dmol

pdb_path = "/content/drive/MyDrive/result/C87A_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb"

with open(pdb_path, "r") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=700, height=500)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [41]:
import py3Dmol

pdb_path = "/content/drive/MyDrive/result/R81E_unrelaxed_rank_001_alphafold2_ptm_model_3_seed_000.pdb"

with open(pdb_path, "r") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=700, height=500)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [42]:
import py3Dmol

files = {
    "wt": "/content/drive/MyDrive/result/hLcn2_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb",
    "c87a": "/content/drive/MyDrive/result/C87A_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb",
    "r81e": "/content/drive/MyDrive/result/R81E_unrelaxed_rank_001_alphafold2_ptm_model_3_seed_000.pdb"
}

view = py3Dmol.view(width=900, height=650)

for path in files.values():
    with open(path, "r") as f:
        pdb_data = f.read()
    view.addModel(pdb_data, "pdb")

# cartoon
view.setStyle({"model": 0}, {"cartoon": {"color": "blue"}})
view.setStyle({"model": 1}, {"cartoon": {"color": "red"}})
view.setStyle({"model": 2}, {"cartoon": {"color": "green"}})

# residue 81
view.addStyle({"model": 0, "resi": 81}, {"stick": {"color": "yellow"}})
view.addStyle({"model": 1, "resi": 81}, {"stick": {"color": "yellow"}})
view.addStyle({"model": 2, "resi": 81}, {"stick": {"color": "yellow"}})

# residue 87
view.addStyle({"model": 0, "resi": 87}, {"stick": {"color": "magenta"}})
view.addStyle({"model": 1, "resi": 87}, {"stick": {"color": "magenta"}})
view.addStyle({"model": 2, "resi": 87}, {"stick": {"color": "magenta"}})

view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Instructions <a name="Instructions"></a>
**Quick start**
1. Upload your single fasta files to a folder in your Google Drive
2. Define path to the fold containing the fasta files (`input_dir`) define an outdir (`output_dir`)
3. Press "Runtime" -> "Run all".

**Result zip file contents**

At the end of the job a all results `jobname.result.zip` will be uploaded to your (`output_dir`) Google Drive. Each zip contains one protein.

1. PDB formatted structures sorted by avg. pIDDT. (unrelaxed and relaxed if `use_amber` is enabled).
2. Plots of the model quality.
3. Plots of the MSA coverage.
4. Parameter log file.
5. A3M formatted input MSA.
6. BibTeX file with citations for all used tools and databases.


**Troubleshooting**
* Check that the runtime type is set to GPU at "Runtime" -> "Change runtime type".
* Try to restart the session "Runtime" -> "Factory reset runtime".
* Check your input sequence.

**Known issues**
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Your browser can block the pop-up for downloading the result file. You can choose the `save_to_google_drive` option to upload to Google Drive instead or manually download the result file: Click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

**Limitations**
* Computing resources: Our MMseqs2 API can handle ~20-50k requests per day.
* MSAs: MMseqs2 is very precise and sensitive but might find less hits compared to HHblits/HMMer searched against BFD or Mgnify.
* We recommend to additionally use the full [AlphaFold2 pipeline](https://github.com/deepmind/alphafold).

**Description of the plots**
*   **Number of sequences per position** - We want to see at least 30 sequences per position, for best performance, ideally 100 sequences.
*   **Predicted lDDT per position** - model confidence (out of 100) at each position. The higher the better.
*   **Predicted Alignment Error** - For homooligomers, this could be a useful metric to assess how confident the model is about the interface. The lower the better.

**Bugs**
- If you encounter any bugs, please report the issue to https://github.com/sokrypton/ColabFold/issues

**License**

The source code of ColabFold is licensed under [MIT](https://raw.githubusercontent.com/sokrypton/ColabFold/main/LICENSE). Additionally, this notebook uses AlphaFold2 source code and its parameters licensed under [Apache 2.0](https://raw.githubusercontent.com/deepmind/alphafold/main/LICENSE) and  [CC BY 4.0](https://creativecommons.org/licenses/by-sa/4.0/) respectively. Read more about the AlphaFold license [here](https://github.com/deepmind/alphafold).

**Acknowledgments**
- We thank the AlphaFold team for developing an excellent model and open sourcing the software.

- Do-Yoon Kim for creating the ColabFold logo.

- A colab by Sergey Ovchinnikov ([@sokrypton](https://twitter.com/sokrypton)), Milot Mirdita ([@milot_mirdita](https://twitter.com/milot_mirdita)) and Martin Steinegger ([@thesteinegger](https://twitter.com/thesteinegger)).
